<a href="https://colab.research.google.com/github/afif-af/Natural-Language-Processing/blob/main/Gated_Reccurent_Units_from_Scratch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np

def sigmoid(x):
  return 1/(1+np.exp(-x))

def tanh(x):
  return np.tanh(x)

In [ ]:
class GRUCell:
  def __init__(self, input_size, hidden_size):
    self.input_size=input_size
    self.hidden_size=hidden_size

    self.Wz =np.random.randn(hidden_size, input_size) * 0.1
    self.Uz =np.random.randn(hidden_size, hidden_size) * 0.1
    self.bz =np.zeros((hidden_size, 1))

    self.Wr =np.random.randn(hidden_size, input_size) * 0.1
    self.Ur =np.random.randn(hidden_size, hidden_size) * 0.1
    self.br =np.zeros((hidden_size, 1))

    self.Wh =np.random.randn(hidden_size, input_size) * 0.1
    self.Uh =np.random.randn(hidden_size, hidden_size) *0.1
    self.bh =np.random.randn(hidden_size, 1)

  def forward(self, x, h_prev):
    z =sigmoid(
        np.dot(self.Wz, x) + np.dot(self.Uz, h_prev) + self.bz
    )
    r =sigmoid(
        np.dot(self.Wr, x) + np.dot(self.Ur, h_prev) + self.br
    )
    h_tilda =tanh(
        np.dot(self.Wh, x) + np.dot(self.Uh, r * h_prev) +self.bh
    )
    h = (1-z) * h_prev + z * h_tilda

    return h, z, r, h_tilda

In [ ]:
class GRU:
  def __init__(self, input_size, hidden_size):
    self.hidden_size =hidden_size
    self.cell =GRUCell(input_size, hidden_size)

  def forward(self,inputs):
    h =np.zeros((self.hidden_size, 1))
    hidden_states =[]

    for t in range(len(inputs)):
      x =inputs[t].reshape(-1, 1)
      h, z,r ,h_tilda =self.cell.forward(x, h)
      hidden_states.append(h)

      print(f"\nTime Step {t+1}")
      print("Input x:")
      print(x)

      print("Update Gate z:")
      print(z)

      print("Reset Gate r:")
      print(r)

      print("Candidate Hidden State h_tilda:")
      print(h_tilda)

      print("Final Hidden State h:")
      print(h)

    return hidden_states

In [ ]:
sequence =np.array([
    [0.5],
    [0.2],
    [0.1],
    [0.7]
])

In [ ]:
input_size =1
hidden_size=3

gru =GRU(input_size, hidden_size)

outputs =gru.forward(sequence)

print("\n=================")
print("Final Output Hidden State")
print("===================")
print(outputs[-1])



Time Step 1
Input x:
[[0.5]]
Update Gate z:
[[0.51084541]
 [0.47799615]
 [0.49287386]]
Reset Gate r:
[[0.48628047]
 [0.50522182]
 [0.49810641]]
Candidate Hidden State h_tilda:
[[-0.99323561]
 [-0.50311117]
 [ 0.30384489]]
Final Hidden State h:
[[-0.50738985]
 [-0.2404852 ]
 [ 0.1497572 ]]

Time Step 2
Input x:
[[0.2]]
Update Gate z:
[[0.49807537]
 [0.4695561 ]
 [0.53265113]]
Reset Gate r:
[[0.50258961]
 [0.49288691]
 [0.505858  ]]
Candidate Hidden State h_tilda:
[[-0.99295752]
 [-0.51872628]
 [ 0.2978872 ]]
Final Hidden State h:
[[-0.74923915]
 [-0.371135  ]
 [ 0.22865881]]

Time Step 3
Input x:
[[0.1]]
Update Gate z:
[[0.49260646]
 [0.46316215]
 [0.55157318]]
Reset Gate r:
[[0.50955436]
 [0.48789057]
 [0.50966365]]
Candidate Hidden State h_tilda:
[[-0.99287762]
 [-0.52893228]
 [ 0.29679474]]
Final Hidden State h:
[[-0.86925703]
 [-0.44422073]
 [ 0.26624076]]

Time Step 4
Input x:
[[0.7]]
Update Gate z:
[[0.50384862]
 [0.43149354]
 [0.55209058]]
Reset Gate r:
[[0.49540932]
 [0.4924212

In [ ]:
import numpy as np

def sigmoid(x):
    return 1 / (1 + np.exp(-x))

def dsigmoid(y):
    return y * (1 - y)

def tanh(x):
    return np.tanh(x)

def dtanh(y):
    return 1 - y**2


In [ ]:
class GRU:

    def __init__(self, input_size, hidden_size, output_size, lr=0.01):

        self.input_size = input_size
        self.hidden_size = hidden_size
        self.output_size = output_size
        self.lr = lr

        # =================================================
        # UPDATE GATE
        # =================================================
        self.Wz = np.random.randn(hidden_size, input_size) * 0.1
        self.Uz = np.random.randn(hidden_size, hidden_size) * 0.1
        self.bz = np.zeros((hidden_size, 1))

        # =================================================
        # RESET GATE
        # =================================================
        self.Wr = np.random.randn(hidden_size, input_size) * 0.1
        self.Ur = np.random.randn(hidden_size, hidden_size) * 0.1
        self.br = np.zeros((hidden_size, 1))

        # =================================================
        # CANDIDATE HIDDEN
        # =================================================
        self.Wh = np.random.randn(hidden_size, input_size) * 0.1
        self.Uh = np.random.randn(hidden_size, hidden_size) * 0.1
        self.bh = np.zeros((hidden_size, 1))

        # =================================================
        # OUTPUT LAYER
        # =================================================
        self.Wy = np.random.randn(output_size, hidden_size) * 0.1
        self.by = np.zeros((output_size, 1))


    # =====================================================
    # FORWARD
    # =====================================================

    def forward(self, inputs):

        h_prev = np.zeros((self.hidden_size, 1))

        self.cache = []

        for x in inputs:

            x = x.reshape(-1, 1)

            # ---------------------------------------------
            # UPDATE GATE
            # ---------------------------------------------
            z = sigmoid(
                self.Wz @ x +
                self.Uz @ h_prev +
                self.bz
            )

            # ---------------------------------------------
            # RESET GATE
            # ---------------------------------------------
            r = sigmoid(
                self.Wr @ x +
                self.Ur @ h_prev +
                self.br
            )

            # ---------------------------------------------
            # CANDIDATE HIDDEN
            # ---------------------------------------------
            h_tilda = tanh(
                self.Wh @ x +
                self.Uh @ (r * h_prev) +
                self.bh
            )

            # ---------------------------------------------
            # FINAL HIDDEN
            # ---------------------------------------------
            h = (1 - z) * h_prev + z * h_tilda

            self.cache.append({
                "x": x,
                "h_prev": h_prev,
                "z": z,
                "r": r,
                "h_tilda": h_tilda,
                "h": h
            })

            h_prev = h

        # =================================================
        # OUTPUT
        # =================================================
        y = self.Wy @ h + self.by

        return y, h


    # =====================================================
    # LOSS
    # =====================================================

    def mse_loss(self, y_pred, y_true):
        return np.mean((y_pred - y_true) ** 2)


    # =====================================================
    # BACKWARD
    # =====================================================

    def backward(self, y_pred, y_true):

        # =================================================
        # INITIALIZE GRADIENTS
        # =================================================

        dWz = np.zeros_like(self.Wz)
        dUz = np.zeros_like(self.Uz)
        dbz = np.zeros_like(self.bz)

        dWr = np.zeros_like(self.Wr)
        dUr = np.zeros_like(self.Ur)
        dbr = np.zeros_like(self.br)

        dWh = np.zeros_like(self.Wh)
        dUh = np.zeros_like(self.Uh)
        dbh = np.zeros_like(self.bh)

        dWy = np.zeros_like(self.Wy)
        dby = np.zeros_like(self.by)

        # =================================================
        # OUTPUT GRADIENT
        # =================================================

        dy = 2 * (y_pred - y_true)

        last_h = self.cache[-1]["h"]

        dWy += dy @ last_h.T
        dby += dy

        dh_next = self.Wy.T @ dy

        # =================================================
        # BPTT
        # =================================================

        for t in reversed(range(len(self.cache))):

            cache = self.cache[t]

            x = cache["x"]
            h_prev = cache["h_prev"]
            z = cache["z"]
            r = cache["r"]
            h_tilda = cache["h_tilda"]
            h = cache["h"]

            # ---------------------------------------------
            # h gradient
            # ---------------------------------------------
            dh = dh_next

            # ---------------------------------------------
            # h = (1-z)*h_prev + z*h_tilda
            # ---------------------------------------------
            dz = dh * (h_tilda - h_prev)

            dh_tilda = dh * z

            dh_prev = dh * (1 - z)

            # ---------------------------------------------
            # tanh backward
            # ---------------------------------------------
            da_h = dh_tilda * dtanh(h_tilda)

            dWh += da_h @ x.T
            dUh += da_h @ (r * h_prev).T
            dbh += da_h

            # ---------------------------------------------
            # reset gate gradient
            # ---------------------------------------------
            drhp = self.Uh.T @ da_h

            dr = drhp * h_prev

            dr_raw = dr * dsigmoid(r)

            dWr += dr_raw @ x.T
            dUr += dr_raw @ h_prev.T
            dbr += dr_raw

            # ---------------------------------------------
            # update gate backward
            # ---------------------------------------------
            dz_raw = dz * dsigmoid(z)

            dWz += dz_raw @ x.T
            dUz += dz_raw @ h_prev.T
            dbz += dz_raw

            # ---------------------------------------------
            # previous hidden gradient
            # ---------------------------------------------
            dh_prev += (
                self.Uz.T @ dz_raw
                + self.Ur.T @ dr_raw
                + (self.Uh.T @ da_h) * r
            )

            dh_next = dh_prev


        # =================================================
        # STORE GRADIENTS
        # =================================================

        self.grads = {
            "Wz": dWz,
            "Uz": dUz,
            "bz": dbz,

            "Wr": dWr,
            "Ur": dUr,
            "br": dbr,

            "Wh": dWh,
            "Uh": dUh,
            "bh": dbh,

            "Wy": dWy,
            "by": dby
        }


    # =====================================================
    # SGD OPTIMIZER
    # =====================================================

    def step(self):

        self.Wz -= self.lr * self.grads["Wz"]
        self.Uz -= self.lr * self.grads["Uz"]
        self.bz -= self.lr * self.grads["bz"]

        self.Wr -= self.lr * self.grads["Wr"]
        self.Ur -= self.lr * self.grads["Ur"]
        self.br -= self.lr * self.grads["br"]

        self.Wh -= self.lr * self.grads["Wh"]
        self.Uh -= self.lr * self.grads["Uh"]
        self.bh -= self.lr * self.grads["bh"]

        self.Wy -= self.lr * self.grads["Wy"]
        self.by -= self.lr * self.grads["by"]


    # =====================================================
    # TRAIN
    # =====================================================

    def train(self, X, Y, epochs=100):

        for epoch in range(epochs):

            total_loss = 0

            for x_seq, y_true in zip(X, Y):

                y_true = y_true.reshape(-1, 1)

                # Forward
                y_pred, _ = self.forward(x_seq)

                # Loss
                loss = self.mse_loss(y_pred, y_true)

                # Backward
                self.backward(y_pred, y_true)

                # SGD update
                self.step()

                total_loss += loss

            if epoch % 10 == 0:
                print(f"Epoch {epoch} Loss = {total_loss:.6f}")


# =========================================================
# DATASET
# =========================================================

X = [
    np.array([[0.1], [0.2], [0.3]]),
    np.array([[0.2], [0.3], [0.4]]),
    np.array([[0.3], [0.4], [0.5]])
]

Y = [
    np.array([0.4]),
    np.array([0.5]),
    np.array([0.6])
]

# =========================================================
# CREATE MODEL
# =========================================================

model = GRU(
    input_size=1,
    hidden_size=8,
    output_size=1,
    lr=0.01
)

# =========================================================
# TRAIN
# =========================================================

model.train(X, Y, epochs=200)

# =========================================================
# TEST
# =========================================================

test_seq = np.array([
    [0.5],
    [0.6],
    [0.7]
])

pred, hidden = model.forward(test_seq)

print("\nPrediction:")
print(pred)

Epoch 0 Loss = 0.738716
Epoch 10 Loss = 0.216391
Epoch 20 Loss = 0.072469
Epoch 30 Loss = 0.033282
Epoch 40 Loss = 0.022758
Epoch 50 Loss = 0.019968
Epoch 60 Loss = 0.019234
Epoch 70 Loss = 0.019036
Epoch 80 Loss = 0.018972
Epoch 90 Loss = 0.018938
Epoch 100 Loss = 0.018911
Epoch 110 Loss = 0.018884
Epoch 120 Loss = 0.018857
Epoch 130 Loss = 0.018828
Epoch 140 Loss = 0.018800
Epoch 150 Loss = 0.018771
Epoch 160 Loss = 0.018742
Epoch 170 Loss = 0.018714
Epoch 180 Loss = 0.018685
Epoch 190 Loss = 0.018656

Prediction:
[[0.51527624]]
